In [308]:
import pandas as pd
import numpy as np

In [309]:
nominal = pd.read_csv("../data/raw/Nominal House Price indices.csv")
real = pd.read_csv("../data/raw/Real House Price Index.csv")
rent = pd.read_csv("../data/raw/Rent Prices.csv")
price_income = pd.read_csv("../data/raw/Price to income ratio.csv")
price_rent = pd.read_csv("../data/raw/Price to rent ratio.csv")

In [310]:
print("Nominal:", nominal.shape)
print("Real:", real.shape)
print("Rent:", rent.shape)
print("Price to Income:", price_income.shape)
print("Price to Rent:", price_rent.shape)

Nominal: (139, 26)
Real: (139, 26)
Rent: (144, 26)
Price to Income: (139, 26)
Price to Rent: (139, 26)


In [311]:
nominal[["REF_AREA", "Reference area", "TIME_PERIOD", "OBS_VALUE", "Measure"]].head()

,REF_AREA,Reference area,TIME_PERIOD,OBS_VALUE,Measure
0,AUT,Austria,2010,77.3225,Nominal house price indices
1,AUT,Austria,2011,81.4725,Nominal house price indices
2,AUT,Austria,2013,91.1950,Nominal house price indices
3,AUT,Austria,2014,94.6750,Nominal house price indices
4,AUT,Austria,2016,106.7050,Nominal house price indices


In [312]:
def clean_oecd_file(df, new_value_name):
    """
    This function cleans one OECD housing file.
    It keeps only the important columns, renames them clearly,
    fixes data types, and returns a clean table.
    """
    
    # Keep only the columns we need
    clean_df = df[["REF_AREA", "Reference area", "TIME_PERIOD", "OBS_VALUE"]].copy()
    
    # Rename columns to simple names
    clean_df = clean_df.rename(columns={
        "REF_AREA": "country_code",
        "Reference area": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": new_value_name
    })
    
    # Make sure year is a whole number
    clean_df["year"] = clean_df["year"].astype(int)
    
    # Make sure the indicator value is numeric
    clean_df[new_value_name] = pd.to_numeric(clean_df[new_value_name], errors="coerce")
    
    # Remove duplicate rows if any exist
    clean_df = clean_df.drop_duplicates()
    
    # Sort data nicely
    clean_df = clean_df.sort_values(["country", "year"]).reset_index(drop=True)
    
    return clean_df

In [313]:
nominal_clean = clean_oecd_file(nominal, "nominal_house_price_index")
real_clean = clean_oecd_file(real, "real_house_price_index")
rent_clean = clean_oecd_file(rent, "rent_price_index")
price_income_clean = clean_oecd_file(price_income, "price_to_income_ratio")
price_rent_clean = clean_oecd_file(price_rent, "price_to_rent_ratio")

In [314]:
nominal_clean.head()

,country_code,country,year,nominal_house_price_index
0,AUT,Austria,2010,77.3225
1,AUT,Austria,2011,81.4725
2,AUT,Austria,2012,86.8625
3,AUT,Austria,2013,91.1950
4,AUT,Austria,2014,94.6750


In [315]:
datasets = {
    "nominal_clean": nominal_clean,
    "real_clean": real_clean,
    "rent_clean": rent_clean,
    "price_income_clean": price_income_clean,
    "price_rent_clean": price_rent_clean
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


nominal_clean
country_code                 0
country                      0
year                         0
nominal_house_price_index    0
dtype: int64

real_clean
country_code              0
country                   0
year                      0
real_house_price_index    0
dtype: int64

rent_clean
country_code        0
country             0
year                0
rent_price_index    0
dtype: int64

price_income_clean
country_code             0
country                  0
year                     0
price_to_income_ratio    0
dtype: int64

price_rent_clean
country_code           0
country                0
year                   0
price_to_rent_ratio    0
dtype: int64


In [316]:
housing_master = nominal_clean.merge(
    real_clean,
    on=["country_code", "country", "year"],
    how="outer"
)

housing_master = housing_master.merge(
    rent_clean,
    on=["country_code", "country", "year"],
    how="outer"
)

housing_master = housing_master.merge(
    price_income_clean,
    on=["country_code", "country", "year"],
    how="outer"
)

housing_master = housing_master.merge(
    price_rent_clean,
    on=["country_code", "country", "year"],
    how="outer"
)

In [317]:
#sorting- for orderly file organisation
housing_master = housing_master.sort_values(["country", "year"]).reset_index(drop=True)

In [318]:
housing_master.head(20)

,country_code,country,year,nominal_house_price_index,real_house_price_index,rent_price_index,price_to_income_ratio,price_to_rent_ratio
0,AUT,Austria,2010,77.3225,85.976098,82.808831,82.870314,93.368295
1,AUT,Austria,2011,81.4725,87.855382,85.556472,86.435537,95.219128
2,AUT,Austria,2012,86.8625,91.544150,89.325850,88.277640,97.242014
3,AUT,Austria,2013,91.1950,94.204279,92.101098,93.843900,99.016565
4,AUT,Austria,2014,94.6750,95.971274,95.760014,95.740826,98.867577
5,AUT,Austria,2015,100.0000,100.000000,100.000000,100.000000,100.000000
6,AUT,Austria,2016,106.7050,105.264721,103.082848,103.761195,103.511104
7,AUT,Austria,2017,112.1350,108.565297,107.340474,106.756441,104.469993
8,AUT,Austria,2018,118.8300,112.616959,111.331500,109.720565,106.727748
9,AUT,Austria,2019,125.9675,117.317830,114.639251,114.615839,109.873116


In [319]:
housing_master.info()

<class 'pandas.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   country_code               144 non-null    str    
 1   country                    144 non-null    str    
 2   year                       144 non-null    int64  
 3   nominal_house_price_index  139 non-null    float64
 4   real_house_price_index     139 non-null    float64
 5   rent_price_index           144 non-null    float64
 6   price_to_income_ratio      139 non-null    float64
 7   price_to_rent_ratio        139 non-null    float64
dtypes: float64(5), int64(1), str(2)
memory usage: 9.1 KB


In [320]:
housing_master.shape

(144, 8)

In [321]:
coverage = housing_master.groupby("country").agg(
    first_year=("year", "min"),
    last_year=("year", "max"),
    number_of_years=("year", "count")
).reset_index()

coverage

,country,first_year,last_year,number_of_years
0,Austria,2010,2025,16
1,France,2010,2025,16
2,Germany,2010,2025,16
3,Italy,2010,2025,16
4,Netherlands,2010,2025,16
5,Poland,2010,2025,16
6,Portugal,2010,2025,16
7,Spain,2010,2025,16
8,Sweden,2010,2025,16


In [322]:
housing_master = housing_master.dropna(subset=["nominal_house_price_index"])

In [323]:
coverage

,country,first_year,last_year,number_of_years
0,Austria,2010,2025,16
1,France,2010,2025,16
2,Germany,2010,2025,16
3,Italy,2010,2025,16
4,Netherlands,2010,2025,16
5,Poland,2010,2025,16
6,Portugal,2010,2025,16
7,Spain,2010,2025,16
8,Sweden,2010,2025,16


In [324]:
# percentage growth in figures over the years

housing_master["nominal_price_yoy_growth_pct"] = (
    housing_master.groupby("country")["nominal_house_price_index"]
    .pct_change() * 100
)

In [325]:
housing_master["real_price_yoy_growth_pct"] = (
    housing_master.groupby("country")["real_house_price_index"]
    .pct_change() * 100
)

In [326]:
housing_master["rent_yoy_growth_pct"] = (
    housing_master.groupby("country")["rent_price_index"]
    .pct_change() * 100
)

In [327]:
#total growth rate since 2010(%)

housing_master["nominal_price_total_growth_pct"] = (
    housing_master.groupby("country")["nominal_house_price_index"]
    .transform(lambda x: (x / x.iloc[0] - 1) * 100)
)

housing_master["rent_total_growth_pct"] = (
    housing_master.groupby("country")["rent_price_index"]
    .transform(lambda x: (x / x.iloc[0] - 1) * 100)
)

In [328]:
#filtering with latest year

latest_year = housing_master["year"].max()

latest_data = housing_master[housing_master["year"] == latest_year].copy()

latest_data

,country_code,country,year,nominal_house_price_index,real_house_price_index,rent_price_index,price_to_income_ratio,price_to_rent_ratio,nominal_price_yoy_growth_pct,real_price_yoy_growth_pct,rent_yoy_growth_pct,nominal_price_total_growth_pct,rent_total_growth_pct
31,FRA,France,2025,127.025000,103.845669,109.727217,93.422266,115.768763,0.633789,-0.031615,2.320094,26.298782,16.526175
47,DEU,Germany,2025,152.688382,116.486076,117.461050,107.938817,129.994960,3.176805,0.608372,2.109189,82.051007,25.241997
63,ITA,Italy,2025,116.200000,94.764904,112.607734,88.469999,103.187536,4.028648,2.544303,3.818092,-1.608806,18.253404
111,PRT,Portugal,2025,263.847500,206.377294,136.370575,166.744226,193.399279,17.558145,14.669521,5.302943,145.765317,50.086203


In [329]:
common_year_data = housing_master.dropna(subset=[
    "nominal_house_price_index",
    "real_house_price_index",
    "rent_price_index",
    "price_to_income_ratio",
    "price_to_rent_ratio"
])

latest_common_year = common_year_data["year"].max()

latest_data = common_year_data[common_year_data["year"] == latest_common_year].copy()

latest_common_year

np.int64(2025)

In [330]:
# to normalize values using scores from 1-100

def normalize_positive(series):
    return (series - series.min()) / (series.max() - series.min()) * 100

def normalize_negative(series):
    return (series.max() - series) / (series.max() - series.min()) * 100

In [331]:
#investment score calculation

latest_data["growth_score"] = normalize_positive(latest_data["nominal_price_total_growth_pct"])
latest_data["rent_score"] = normalize_positive(latest_data["rent_total_growth_pct"])
latest_data["affordability_score"] = normalize_negative(latest_data["price_to_income_ratio"])
latest_data["value_score"] = normalize_negative(latest_data["price_to_rent_ratio"])

latest_data["investment_score"] = (
    latest_data["growth_score"] * 0.35 +
    latest_data["rent_score"] * 0.25 +
    latest_data["affordability_score"] * 0.25 +
    latest_data["value_score"] * 0.15
)

In [332]:
#final investment ranking (where should you buy property in europe)

investment_ranking = latest_data[[
    "country",
    "year",
    "nominal_price_total_growth_pct",
    "rent_total_growth_pct",
    "price_to_income_ratio",
    "price_to_rent_ratio",
    "investment_score"
]].sort_values("investment_score", ascending=False)

investment_ranking

,country,year,nominal_price_total_growth_pct,rent_total_growth_pct,price_to_income_ratio,price_to_rent_ratio,investment_score
111,Portugal,2025,145.765317,50.086203,166.744226,193.399279,60.000000
47,Germany,2025,82.051007,25.241997,107.938817,129.994960,55.685586
31,France,2025,26.298782,16.526175,93.422266,115.768763,42.954142
63,Italy,2025,-1.608806,18.253404,88.469999,103.187536,41.286671


In [333]:
housing_master.to_csv("../data/cleaned/oecd_housing_master_cleaned.csv", index=False)
investment_ranking.to_csv("../data/cleaned/investment_ranking.csv", index=False)

Part 2

In [334]:
wages = pd.read_csv("../data/raw/oecd_average_annual_wages.csv")
income = pd.read_csv("../data/raw/oecd_household_disposable_income.csv")
interest = pd.read_csv("../data/raw/oecd_long_term_interest_rate.csv")
population = pd.read_csv("../data/raw/oecd_population.csv")

filtering

In [335]:
#keep only current prices
wages = wages[wages["Price base"] == "Current prices"].copy()

#keeping only total age
population = population[population["Age"] == "Total"].copy()

#avoid duplicates using first available scenario
interest = interest[interest["Scenario"] == interest["Scenario"].unique()[0]].copy()

In [336]:
print("wages:", wages.shape)
print("income:", income.shape)
print("interest:", interest.shape)
print("population:", population.shape)

wages: (135, 30)
income: (135, 64)
interest: (153, 24)
population: (135, 26)


In [337]:
wages.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,MEASURE,Measure,UNIT_MEASURE,Unit of measure,...,OBS_VALUE,Observation value,BASE_PER,Base period,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
135,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,AUT,Austria,WG,Wages,EUR,Euro,...,36382.634,NaN,NaN,NaN,A,Normal value,0,Units,0,Zero
136,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,AUT,Austria,WG,Wages,EUR,Euro,...,37127.955,NaN,NaN,NaN,A,Normal value,0,Units,0,Zero
137,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,AUT,Austria,WG,Wages,EUR,Euro,...,38215.019,NaN,NaN,NaN,A,Normal value,0,Units,0,Zero
138,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,AUT,Austria,WG,Wages,EUR,Euro,...,39035.153,NaN,NaN,NaN,A,Normal value,0,Units,0,Zero
139,DATAFLOW,OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0),Average annual wages,I,AUT,Austria,WG,Wages,EUR,Euro,...,39974.325,NaN,NaN,NaN,A,Normal value,0,Units,0,Zero


In [338]:
print(wages.columns)
print(income.columns)
print(interest.columns)
print(population.columns)

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA',
       'Reference area', 'MEASURE', 'Measure', 'UNIT_MEASURE',
       'Unit of measure', 'PAY_PERIOD', 'Pay period', 'PRICE_BASE',
       'Price base', 'AGGREGATION_OPERATION', 'Aggregation operation', 'SEX',
       'Sex', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value',
       'BASE_PER', 'Base period', 'OBS_STATUS', 'Observation status',
       'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals'],
      dtype='str')
Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ',
       'Frequency of observation', 'REF_AREA', 'Reference area', 'MEASURE',
       'Measure', 'UNIT_MEASURE', 'Unit of measure', 'CHAPTER', 'Chapter',
       'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value',
       'ADJUSTMENT', 'Adjustment', 'COUNTERPART_AREA', 'Counterpart area',
       'SECTOR', 'Institutional sector', 'COUNTERPART_SECTOR',
       'Counterpart institutional sector', 'CONSOLID

In [339]:
def clean_oecd_file(df, value_column_name):
    df_clean = df[["REF_AREA", "Reference area", "TIME_PERIOD", "OBS_VALUE"]].copy()

    df_clean = df_clean.rename(columns={
        "REF_AREA": "country_code",
        "Reference area": "country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": value_column_name
    })

    df_clean["country_code"] = df_clean["country_code"].astype(str).str.strip()
    df_clean["country"] = df_clean["country"].astype(str).str.strip()
    df_clean["year"] = pd.to_numeric(df_clean["year"], errors="coerce").astype(int)
    df_clean[value_column_name] = pd.to_numeric(df_clean[value_column_name], errors="coerce")

    df_clean = df_clean.drop_duplicates(subset=["country_code", "country", "year"])

    return df_clean.sort_values(["country", "year"]).reset_index(drop=True)

In [340]:
wages_clean = clean_oecd_file(wages, "average_annual_wage")
income_clean = clean_oecd_file(income, "household_disposable_income")
interest_clean = clean_oecd_file(interest, "interest_rate")
population_clean = clean_oecd_file(population, "population")

In [341]:
#checking duplicates before merging

for name, df in {
    "wages_clean": wages_clean,
    "income_clean": income_clean,
    "interest_clean": interest_clean,
    "population_clean": population_clean
}.items():
    print(name, df.duplicated(subset=["country_code", "country", "year"]).sum())

wages_clean 0
income_clean 0
interest_clean 0
population_clean 0


In [342]:
housing_master = pd.read_csv("../data/cleaned/oecd_housing_master_cleaned.csv")

In [343]:
housing_master.duplicated(subset=["country_code", "country", "year"]).sum()

np.int64(0)

In [344]:
housing_master = housing_master.merge(
    wages_clean,
    on=["country_code", "country", "year"],
    how="left"
)

housing_master = housing_master.merge(
    income_clean,
    on=["country_code", "country", "year"],
    how="left"
)

housing_master = housing_master.merge(
    interest_clean,
    on=["country_code", "country", "year"],
    how="left"
)

housing_master = housing_master.merge(
    population_clean,
    on=["country_code", "country", "year"],
    how="left"
)

In [345]:
housing_master[housing_master["country"] == "Austria"].head(20)

,country_code,country,year,nominal_house_price_index,real_house_price_index,rent_price_index,price_to_income_ratio,price_to_rent_ratio,nominal_price_yoy_growth_pct,real_price_yoy_growth_pct,rent_yoy_growth_pct,nominal_price_total_growth_pct,rent_total_growth_pct,average_annual_wage,household_disposable_income,interest_rate,population
0,AUT,Austria,2010,77.3225,85.976098,82.808831,82.870314,93.368295,NaN,NaN,NaN,0.000000,0.000000,36382.634,25697.230613,3.226383,8357367.0
1,AUT,Austria,2011,81.4725,87.855382,85.556472,86.435537,95.219128,5.367131,2.185821,3.318053,5.367131,3.318053,37127.955,26186.947975,3.319658,8384901.0
2,AUT,Austria,2012,86.8625,91.544150,89.325850,88.277640,97.242014,6.615729,4.198682,4.405720,12.337935,7.869956,38215.019,27163.483183,2.371175,8420328.0
3,AUT,Austria,2013,91.1950,94.204279,92.101098,93.843900,99.016565,4.987768,2.905843,3.106881,17.941091,11.221348,39035.153,27214.780864,2.010758,8472176.0
4,AUT,Austria,2014,94.6750,95.971274,95.760014,95.740826,98.867577,3.815999,1.875705,3.972717,22.441721,15.639857,39974.325,27687.507113,1.487292,8538350.0
5,AUT,Austria,2015,100.0000,100.000000,100.000000,100.000000,100.000000,5.624505,4.197846,4.427721,29.328462,20.760067,40847.443,27843.146991,0.747042,8618709.0
6,AUT,Austria,2016,106.7050,105.264721,103.082848,103.761195,103.511104,6.705000,5.264721,3.082848,37.999935,24.482917,41836.305,29236.532684,0.375383,8737663.0
7,AUT,Austria,2017,112.1350,108.565297,107.340474,106.756441,104.469993,5.088796,3.135500,4.130295,45.022471,29.624429,42570.936,29669.293548,0.583283,8790624.0
8,AUT,Austria,2018,118.8300,112.616959,111.331500,109.720565,106.727748,5.970482,3.732005,3.718100,53.681011,34.443994,43587.122,30955.264465,0.686158,8833192.0
9,AUT,Austria,2019,125.9675,117.317830,114.639251,114.615839,109.873116,6.006480,4.174212,2.971083,62.911830,38.438436,44719.413,32303.488573,0.062933,8872344.0


In [346]:
#recalculating growth columns

housing_master = housing_master.sort_values(["country", "year"]).reset_index(drop=True)

housing_master["nominal_price_yoy_growth_pct"] = (
    housing_master.groupby("country")["nominal_house_price_index"].pct_change() * 100
)

housing_master["real_price_yoy_growth_pct"] = (
    housing_master.groupby("country")["real_house_price_index"].pct_change() * 100
)

housing_master["rent_yoy_growth_pct"] = (
    housing_master.groupby("country")["rent_price_index"].pct_change() * 100
)

In [347]:
housing_master[[
    "country",
    "year",
    "nominal_house_price_index",
    "nominal_price_yoy_growth_pct",
    "real_price_yoy_growth_pct",
    "rent_yoy_growth_pct"
]].head(30)

,country,year,nominal_house_price_index,nominal_price_yoy_growth_pct,real_price_yoy_growth_pct,rent_yoy_growth_pct
0,Austria,2010,77.3225,NaN,NaN,NaN
1,Austria,2011,81.4725,5.367131,2.185821,3.318053
2,Austria,2012,86.8625,6.615729,4.198682,4.405720
3,Austria,2013,91.1950,4.987768,2.905843,3.106881
4,Austria,2014,94.6750,3.815999,1.875705,3.972717
5,Austria,2015,100.0000,5.624505,4.197846,4.427721
6,Austria,2016,106.7050,6.705000,5.264721,3.082848
7,Austria,2017,112.1350,5.088796,3.135500,4.130295
8,Austria,2018,118.8300,5.970482,3.732005,3.718100
9,Austria,2019,125.9675,6.006480,4.174212,2.971083


In [348]:
wages_clean.head()

,country_code,country,year,average_annual_wage
0,AUT,Austria,2010,36382.634
1,AUT,Austria,2011,37127.955
2,AUT,Austria,2012,38215.019
3,AUT,Austria,2013,39035.153
4,AUT,Austria,2014,39974.325


In [349]:
income_clean.head()

,country_code,country,year,household_disposable_income
0,AUT,Austria,2010,25697.230613
1,AUT,Austria,2011,26186.947975
2,AUT,Austria,2012,27163.483183
3,AUT,Austria,2013,27214.780864
4,AUT,Austria,2014,27687.507113


In [350]:
#checking for mising values 

print("Wages missing values:")
print(wages_clean.isnull().sum())

print("\nIncome missing values:")
print(income_clean.isnull().sum())

print("\nInterest missing values:")
print(interest_clean.isnull().sum())

print("\nPopulation missing values:")
print(population_clean.isnull().sum())

Wages missing values:
country_code           0
country                0
year                   0
average_annual_wage    0
dtype: int64

Income missing values:
country_code                   0
country                        0
year                           0
household_disposable_income    0
dtype: int64

Interest missing values:
country_code     0
country          0
year             0
interest_rate    0
dtype: int64

Population missing values:
country_code    0
country         0
year            0
population      0
dtype: int64


In [351]:
print("Wages countries:")
print(wages_clean["country"].unique())

print("\nIncome countries:")
print(income_clean["country"].unique())

print("\nInterest countries:")
print(interest_clean["country"].unique())

print("\nPopulation countries:")
print(population_clean["country"].unique())

Wages countries:
<StringArray>
[    'Austria',      'France',     'Germany',       'Italy', 'Netherlands',
      'Poland',    'Portugal',       'Spain',      'Sweden']
Length: 9, dtype: str

Income countries:
<StringArray>
[    'Austria',      'France',     'Germany',       'Italy', 'Netherlands',
      'Poland',    'Portugal',       'Spain',      'Sweden']
Length: 9, dtype: str

Interest countries:
<StringArray>
[    'Austria',      'France',     'Germany',       'Italy', 'Netherlands',
      'Poland',    'Portugal',       'Spain',      'Sweden']
Length: 9, dtype: str

Population countries:
<StringArray>
[    'Austria',      'France',     'Germany',       'Italy', 'Netherlands',
      'Poland',    'Portugal',       'Spain',      'Sweden']
Length: 9, dtype: str


In [352]:
def check_year_coverage(df, value_name):
    return df.groupby("country").agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        number_of_years=("year", "count")
    ).reset_index().assign(dataset=value_name)

wages_coverage = check_year_coverage(wages_clean, "wages")
income_coverage = check_year_coverage(income_clean, "income")
interest_coverage = check_year_coverage(interest_clean, "interest")
population_coverage = check_year_coverage(population_clean, "population")

coverage_summary = pd.concat([
    wages_coverage,
    income_coverage,
    interest_coverage,
    population_coverage
])

coverage_summary

,country,first_year,last_year,number_of_years,dataset
0,Austria,2010,2024,15,wages
1,France,2010,2024,15,wages
2,Germany,2010,2024,15,wages
3,Italy,2010,2024,15,wages
4,Netherlands,2010,2024,15,wages
5,Poland,2010,2024,15,wages
6,Portugal,2010,2024,15,wages
7,Spain,2010,2024,15,wages
8,Sweden,2010,2024,15,wages
0,Austria,2010,2024,15,income


In [353]:
housing_master.head()

,country_code,country,year,nominal_house_price_index,real_house_price_index,rent_price_index,price_to_income_ratio,price_to_rent_ratio,nominal_price_yoy_growth_pct,real_price_yoy_growth_pct,rent_yoy_growth_pct,nominal_price_total_growth_pct,rent_total_growth_pct,average_annual_wage,household_disposable_income,interest_rate,population
0,AUT,Austria,2010,77.3225,85.976098,82.808831,82.870314,93.368295,NaN,NaN,NaN,0.000000,0.000000,36382.634,25697.230613,3.226383,8357367.0
1,AUT,Austria,2011,81.4725,87.855382,85.556472,86.435537,95.219128,5.367131,2.185821,3.318053,5.367131,3.318053,37127.955,26186.947975,3.319658,8384901.0
2,AUT,Austria,2012,86.8625,91.544150,89.325850,88.277640,97.242014,6.615729,4.198682,4.405720,12.337935,7.869956,38215.019,27163.483183,2.371175,8420328.0
3,AUT,Austria,2013,91.1950,94.204279,92.101098,93.843900,99.016565,4.987768,2.905843,3.106881,17.941091,11.221348,39035.153,27214.780864,2.010758,8472176.0
4,AUT,Austria,2014,94.6750,95.971274,95.760014,95.740826,98.867577,3.815999,1.875705,3.972717,22.441721,15.639857,39974.325,27687.507113,1.487292,8538350.0


**Merging datasets**

In [354]:
housing_master.isnull().sum()

country_code                      0
country                           0
year                              0
nominal_house_price_index         0
real_house_price_index            0
rent_price_index                  0
price_to_income_ratio             0
price_to_rent_ratio               0
nominal_price_yoy_growth_pct      9
real_price_yoy_growth_pct         9
rent_yoy_growth_pct               9
nominal_price_total_growth_pct    0
rent_total_growth_pct             0
average_annual_wage               4
household_disposable_income       4
interest_rate                     0
population                        4
dtype: int64

**Creating Business Metrics**

Housing Pressure

In [355]:
#If house prices rise faster than wages, housing becomes harder to afford.
housing_master["housing_pressure"] = (
    housing_master["nominal_house_price_index"] / housing_master["average_annual_wage"]
)

True Affordability

In [356]:
#money households have available after taxes and transfers
housing_master["true_affordability"] = (
    housing_master["household_disposable_income"] / housing_master["nominal_house_price_index"]
)

Financing Pressure

In [357]:
#High house prices + high interest rates = expensive property market
housing_master["financing_pressure"] = (
    housing_master["nominal_house_price_index"] * housing_master["interest_rate"]
)

Population growth

In [358]:
housing_master = housing_master.sort_values(["country", "year"])

housing_master["population_growth"] = (
    housing_master.groupby("country")["population"].pct_change() * 100
)

Demand pressure

In [359]:
#Positive number = population growth is strong compared with price growth
#Negative number = house prices are growing faster than population

housing_master["demand_pressure"] = (
    housing_master["population_growth"] - housing_master["nominal_price_yoy_growth_pct"]
)

In [360]:
housing_master[[
    "country",
    "year",
    "average_annual_wage",
    "household_disposable_income",
    "interest_rate",
    "population",
    "housing_pressure",
    "true_affordability",
    "financing_pressure",
    "population_growth",
    "demand_pressure"
]].head(30)

,country,year,average_annual_wage,household_disposable_income,interest_rate,population,housing_pressure,true_affordability,financing_pressure,population_growth,demand_pressure
0,Austria,2010,36382.634,25697.230613,3.226383,8357367.0,0.002125,332.338331,249.472025,NaN,NaN
1,Austria,2011,37127.955,26186.947975,3.319658,8384901.0,0.002194,321.420700,270.460864,0.329458,-5.037673
2,Austria,2012,38215.019,27163.483183,2.371175,8420328.0,0.002273,312.718183,205.966188,0.422509,-6.193220
3,Austria,2013,39035.153,27214.780864,2.010758,8472176.0,0.002336,298.424046,183.371106,0.615748,-4.372020
4,Austria,2014,39974.325,27687.507113,1.487292,8538350.0,0.002368,292.447923,140.809339,0.781074,-3.034924
5,Austria,2015,40847.443,27843.146991,0.747042,8618709.0,0.002448,278.431470,74.704167,0.941154,-4.683351
6,Austria,2016,41836.305,29236.532684,0.375383,8737663.0,0.002551,273.994027,40.055279,1.380184,-5.324816
7,Austria,2017,42570.936,29669.293548,0.583283,8790624.0,0.002634,264.585487,65.406477,0.606123,-4.482673
8,Austria,2018,43587.122,30955.264465,0.686158,8833192.0,0.002726,260.500416,81.536195,0.484243,-5.486239
9,Austria,2019,44719.413,32303.488573,0.062933,8872344.0,0.002817,256.443039,7.927555,0.443237,-5.563243


In [361]:
housing_master[[
    "average_annual_wage",
    "household_disposable_income",
    "interest_rate",
    "population",
    "housing_pressure",
    "true_affordability",
    "financing_pressure",
    "population_growth",
    "demand_pressure"
]].isnull().sum()

average_annual_wage             4
household_disposable_income     4
interest_rate                   0
population                      4
housing_pressure                4
true_affordability              4
financing_pressure              0
population_growth              13
demand_pressure                13
dtype: int64

In [362]:
required_columns = [
    "nominal_house_price_index",
    "real_house_price_index",
    "rent_price_index",
    "price_to_income_ratio",
    "price_to_rent_ratio",
    "average_annual_wage",
    "household_disposable_income",
    "interest_rate",
    "population",
    "housing_pressure",
    "true_affordability",
    "financing_pressure",
    "population_growth",
    "demand_pressure"
]

complete_data = housing_master.dropna(subset=required_columns)

latest_common_year = complete_data["year"].max()

latest_data = complete_data[complete_data["year"] == latest_common_year].copy()

latest_common_year

np.int64(2024)

**Scoring function**

In [363]:
#for positive metrics- higher is better(rent growth, price growth, affordability)
#for negative metrics- lower is better(interest rate, financing pressure, price-to income ratio)
def normalize_positive(series):
    return (series - series.min()) / (series.max() - series.min()) * 100

def normalize_negative(series):
    return (series.max() - series) / (series.max() - series.min()) * 100

In [364]:
#creating component scores 
latest_data["growth_score"] = normalize_positive(
    latest_data["nominal_price_total_growth_pct"]
)

latest_data["rent_score"] = normalize_positive(
    latest_data["rent_total_growth_pct"]
)

latest_data["affordability_score"] = normalize_positive(
    latest_data["true_affordability"]
)

latest_data["interest_score"] = normalize_negative(
    latest_data["interest_rate"]
)

latest_data["demand_score"] = normalize_positive(
    latest_data["population_growth"]
)

In [365]:
#creating final investment score

latest_data["investment_score"] = (
    latest_data["growth_score"] * 0.25 +
    latest_data["rent_score"] * 0.20 +
    latest_data["affordability_score"] * 0.20 +
    latest_data["interest_score"] * 0.15 +
    latest_data["demand_score"] * 0.20
)

In [366]:
#creating final ranking table - country with highest score ranks first

investment_ranking = latest_data[[
    "country",
    "year",
    "nominal_price_total_growth_pct",
    "rent_total_growth_pct",
    "price_to_income_ratio",
    "price_to_rent_ratio",
    "average_annual_wage",
    "household_disposable_income",
    "interest_rate",
    "population_growth",
    "growth_score",
    "rent_score",
    "affordability_score",
    "interest_score",
    "demand_score",
    "investment_score"
]].sort_values("investment_score", ascending=False)

investment_ranking

,country,year,nominal_price_total_growth_pct,rent_total_growth_pct,price_to_income_ratio,price_to_rent_ratio,average_annual_wage,household_disposable_income,interest_rate,population_growth,growth_score,rent_score,affordability_score,interest_score,demand_score,investment_score
14,Austria,2024,110.882990,70.475455,114.681996,115.514314,56968.032,46831.235422,2.842708,0.538894,100.000000,82.489418,71.018447,80.676977,59.011924,79.605504
107,Portugal,2024,109.058519,42.528023,147.263891,173.251483,23177.059,33735.060377,2.957500,1.162337,98.431266,43.612468,2.972009,77.224316,100.000000,65.508359
45,Germany,2024,76.445673,22.654972,107.844945,128.648436,50256.900,47685.379160,2.321084,0.275336,70.389778,15.967580,88.427903,96.366198,41.684357,61.268342
138,Sweden,2024,78.486446,32.024655,95.494507,110.812999,529659.000,35146.560577,2.200270,0.313943,72.144496,29.001503,60.591909,100.000000,44.222581,59.799323
77,Netherlands,2024,77.872483,43.273569,130.538403,161.189543,58248.347,41485.053617,2.620500,0.655122,71.616592,44.649578,32.300006,87.360477,66.653327,59.728802
29,France,2024,25.503356,13.883960,93.403700,117.710993,44908.538,39420.801801,2.975517,0.263040,26.588071,3.766451,83.496594,76.682417,40.875980,43.777185
92,Poland,2024,95.722469,83.063269,102.752499,124.019024,91625.251,29833.094772,5.525000,-0.358698,86.964537,100.000000,0.000000,0.000000,0.000000,41.741134
123,Spain,2024,19.098811,11.176375,115.697145,144.437446,33043.823,35161.929740,3.153796,1.007061,21.081254,0.000000,37.322898,71.320212,89.791426,41.391210
61,Italy,2024,-5.419136,13.904428,87.042987,102.976149,33148.002,38593.144825,3.707417,-0.053433,0.000000,3.794925,100.000000,54.668600,20.069562,32.973187


In [367]:
def recommendation_label(score):
    if score >= 75:
        return "Strong Buy"
    elif score >= 60:
        return "Attractive"
    elif score >= 45:
        return "Watchlist"
    else:
        return "High Risk / Low Priority"

investment_ranking["recommendation"] = investment_ranking["investment_score"].apply(recommendation_label)

investment_ranking

,country,year,nominal_price_total_growth_pct,rent_total_growth_pct,price_to_income_ratio,price_to_rent_ratio,average_annual_wage,household_disposable_income,interest_rate,population_growth,growth_score,rent_score,affordability_score,interest_score,demand_score,investment_score,recommendation
14,Austria,2024,110.882990,70.475455,114.681996,115.514314,56968.032,46831.235422,2.842708,0.538894,100.000000,82.489418,71.018447,80.676977,59.011924,79.605504,Strong Buy
107,Portugal,2024,109.058519,42.528023,147.263891,173.251483,23177.059,33735.060377,2.957500,1.162337,98.431266,43.612468,2.972009,77.224316,100.000000,65.508359,Attractive
45,Germany,2024,76.445673,22.654972,107.844945,128.648436,50256.900,47685.379160,2.321084,0.275336,70.389778,15.967580,88.427903,96.366198,41.684357,61.268342,Attractive
138,Sweden,2024,78.486446,32.024655,95.494507,110.812999,529659.000,35146.560577,2.200270,0.313943,72.144496,29.001503,60.591909,100.000000,44.222581,59.799323,Watchlist
77,Netherlands,2024,77.872483,43.273569,130.538403,161.189543,58248.347,41485.053617,2.620500,0.655122,71.616592,44.649578,32.300006,87.360477,66.653327,59.728802,Watchlist
29,France,2024,25.503356,13.883960,93.403700,117.710993,44908.538,39420.801801,2.975517,0.263040,26.588071,3.766451,83.496594,76.682417,40.875980,43.777185,High Risk / Low Priority
92,Poland,2024,95.722469,83.063269,102.752499,124.019024,91625.251,29833.094772,5.525000,-0.358698,86.964537,100.000000,0.000000,0.000000,0.000000,41.741134,High Risk / Low Priority
123,Spain,2024,19.098811,11.176375,115.697145,144.437446,33043.823,35161.929740,3.153796,1.007061,21.081254,0.000000,37.322898,71.320212,89.791426,41.391210,High Risk / Low Priority
61,Italy,2024,-5.419136,13.904428,87.042987,102.976149,33148.002,38593.144825,3.707417,-0.053433,0.000000,3.794925,100.000000,54.668600,20.069562,32.973187,High Risk / Low Priority


In [368]:
housing_master.to_csv("../data/cleaned/housing_master_updated.csv", index=False)

investment_ranking.to_csv("../data/cleaned/investment_ranking_updated.csv", index=False)

In [369]:
print("Final master shape:", housing_master.shape)
print("Investment ranking shape:", investment_ranking.shape)
print("Latest common year used:", latest_common_year)

Final master shape: (139, 22)
Investment ranking shape: (9, 17)
Latest common year used: 2024


**Where should you buy Property Now?**

In [370]:
investment_ranking[["country", "investment_score", "recommendation"]]

,country,investment_score,recommendation
14,Austria,79.605504,Strong Buy
107,Portugal,65.508359,Attractive
45,Germany,61.268342,Attractive
138,Sweden,59.799323,Watchlist
77,Netherlands,59.728802,Watchlist
29,France,43.777185,High Risk / Low Priority
92,Poland,41.741134,High Risk / Low Priority
123,Spain,41.391210,High Risk / Low Priority
61,Italy,32.973187,High Risk / Low Priority


Austria has the hghest investment score, showing investment in long-term housing should be prioritised in Austria.